# Notebook 1: Hiperspektral Görüntü Veri Setlerinin Yüklenmesi

Bu notebook'ta şu veri setleri **yüklenir** ve **temel dosyalara kaydedilir**:
1. **Indian Pines**: 145×145 piksel, 200 bant, 16 sınıf
2. **Pavia University**: 610×340 piksel, 103 bant, 9 sınıf
3. **Salinas**: 512×217 piksel, 204 bant, 16 sınıf

Veri setler Kaggle'dan indirilecek ve NumPy formatında kaydedilecektir.

⚠️ **NOT:** Preprocessing, PCA, normalizasyon ve diğer veri işleme adımları için `02_data_preprocessing.ipynb` notebook'unu kullanın.

## 1. Gerekli Kütüphanelerin Kurulumu ve İmport Edilmesi

In [3]:
# Temel kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import io
import warnings
warnings.filterwarnings('ignore')

# Makine öğrenmesi kütüphaneleri
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Görselleştirme ayarları
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Tüm kütüphaneler başarıyla yüklendi!")

Tüm kütüphaneler başarıyla yüklendi!


## 2. Google Drive Montajı

In [4]:
# Google Drive'ı monte et
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/hyperspectral_datasets/'

import os
os.makedirs(BASE_DIR, exist_ok=True)

print(f"✓ Google Drive bağlandı: {BASE_DIR}")

MessageError: User cancelled dfs_ephemeral authorization

## 3. Kaggle API Konfigürasyonu

In [ ]:
# Kaggle API'sini konfigure et
if COLAB:
    # Colab'da kaggle.json dosyasını upload et veya ayarla
    os.environ['KAGGLE_CONFIG_DIR'] = '/root/.kaggle/'
    
# Kaggle kütüphanesi kurulu mu kontrol et
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    print("Kaggle API yüklü.")
except ImportError:
    print("Kaggle API kuruluyor...")
    os.system('pip install -q kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi

## 4. Indian Pines Veri Setini Yükleme

In [ ]:
# Indian Pines veri setini indir
print("Indian Pines veri seti indiriliyor...")

indian_pines_dir = os.path.join(BASE_DIR, 'indian_pines')
os.makedirs(indian_pines_dir, exist_ok=True)

try:
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files('abhijeetgo/indian-pines-hyperspectral-dataset', 
                               path=indian_pines_dir, unzip=True)
    print(f"✓ Indian Pines başarıyla indirildi: {indian_pines_dir}")
except Exception as e:
    print(f"Not: Kaggle API'den indirilemedi. Kullanıcı manuel indirmelidir.")
    print(f"Hata: {e}")

## 5. Indian Pines Veri Setini Yükleme ve Görselleştirme

In [ ]:
# Indian Pines .mat dosyasını yükle
indian_pines_files = [f for f in os.listdir(indian_pines_dir) if f.endswith('.mat')]

if indian_pines_files:
    # Görüntü verilerini yükle
    indian_img_path = os.path.join(indian_pines_dir, 'indian_pines.mat')
    indian_gt_path = os.path.join(indian_pines_dir, 'indian_pines_gt.mat')
    
    if os.path.exists(indian_img_path) and os.path.exists(indian_gt_path):
        indian_img = io.loadmat(indian_img_path)['indian_pines']
        indian_gt = io.loadmat(indian_gt_path)['indian_pines_gt']
        
        print("\n=== INDIAN PINES DATASET ===")
        print(f"Görüntü Boyutu: {indian_img.shape}")
        print(f"Piksel: {indian_img.shape[0]} × {indian_img.shape[1]}")
        print(f"Spektral Bant Sayısı: {indian_img.shape[2]}")
        print(f"\nSınıf Bilgisi:")
        unique_classes = np.unique(indian_gt)
        print(f"Toplam Sınıf Sayısı: {len(unique_classes[unique_classes > 0])}")
        print(f"Etiketli Piksel Sayısı: {np.sum(indian_gt > 0)}")
        print(f"Etiketli Piksel Oranı: {np.sum(indian_gt > 0) / (indian_img.shape[0] * indian_img.shape[1]) * 100:.2f}%")
    else:
        print("Indian Pines .mat dosyaları bulunamadı. Lütfen manuel olarak indirin.")
else:
    print("Indian Pines klasöründe .mat dosyası bulunamadı.")

## 6. Pavia University Veri Setini Yükleme

In [ ]:
# Pavia University veri setini indir
print("Pavia University veri seti indiriliyor...")

pavia_dir = os.path.join(BASE_DIR, 'pavia_university')
os.makedirs(pavia_dir, exist_ok=True)

try:
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files('syamkakarla/pavia-university-hsi', 
                               path=pavia_dir, unzip=True)
    print(f"✓ Pavia University başarıyla indirildi: {pavia_dir}")
except Exception as e:
    print(f"Not: Kaggle API'den indirilemedi. Kullanıcı manuel indirmelidir.")
    print(f"Hata: {e}")

## 7. Pavia University Veri Setini Yükleme ve Görselleştirme

In [ ]:
# Pavia University .mat dosyasını yükle
pavia_files = [f for f in os.listdir(pavia_dir) if f.endswith('.mat')]

if pavia_files:
    pavia_img_path = os.path.join(pavia_dir, 'Pavia.mat')
    pavia_gt_path = os.path.join(pavia_dir, 'Pavia_gt.mat')
    
    if os.path.exists(pavia_img_path) and os.path.exists(pavia_gt_path):
        pavia_img = io.loadmat(pavia_img_path)['pavia']
        pavia_gt = io.loadmat(pavia_gt_path)['pavia_gt']
        
        print("\n=== PAVIA UNIVERSITY DATASET ===")
        print(f"Görüntü Boyutu: {pavia_img.shape}")
        print(f"Piksel: {pavia_img.shape[0]} × {pavia_img.shape[1]}")
        print(f"Spektral Bant Sayısı: {pavia_img.shape[2]}")
        print(f"\nSınıf Bilgisi:")
        unique_classes = np.unique(pavia_gt)
        print(f"Toplam Sınıf Sayısı: {len(unique_classes[unique_classes > 0])}")
        print(f"Etiketli Piksel Sayısı: {np.sum(pavia_gt > 0)}")
        print(f"Etiketli Piksel Oranı: {np.sum(pavia_gt > 0) / (pavia_img.shape[0] * pavia_img.shape[1]) * 100:.2f}%")
    else:
        print("Pavia University .mat dosyaları bulunamadı. Lütfen manuel olarak indirin.")
else:
    print("Pavia University klasöründe .mat dosyası bulunamadı.")

## 8. Salinas Veri Setini Yükleme

In [ ]:
# Salinas veri setini indir
print("Salinas veri seti indiriliyor...")

salinas_dir = os.path.join(BASE_DIR, 'salinas')
os.makedirs(salinas_dir, exist_ok=True)

try:
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files('wangyijialili/salinas', 
                               path=salinas_dir, unzip=True)
    print(f"✓ Salinas başarıyla indirildi: {salinas_dir}")
except Exception as e:
    print(f"Not: Kaggle API'den indirilemedi. Kullanıcı manuel indirmelidir.")
    print(f"Hata: {e}")

## 9. Salinas Veri Setini Yükleme ve Görselleştirme

In [ ]:
# Salinas .mat dosyasını yükle
salinas_files = [f for f in os.listdir(salinas_dir) if f.endswith('.mat')]

if salinas_files:
    salinas_img_path = os.path.join(salinas_dir, 'Salinas_corrected.mat')
    salinas_gt_path = os.path.join(salinas_dir, 'Salinas_gt.mat')
    
    if os.path.exists(salinas_img_path) and os.path.exists(salinas_gt_path):
        salinas_img = io.loadmat(salinas_img_path)['salinas_corrected']
        salinas_gt = io.loadmat(salinas_gt_path)['salinas_gt']
        
        print("\n=== SALINAS DATASET ===")
        print(f"Görüntü Boyutu: {salinas_img.shape}")
        print(f"Piksel: {salinas_img.shape[0]} × {salinas_img.shape[1]}")
        print(f"Spektral Bant Sayısı: {salinas_img.shape[2]}")
        print(f"\nSınıf Bilgisi:")
        unique_classes = np.unique(salinas_gt)
        print(f"Toplam Sınıf Sayısı: {len(unique_classes[unique_classes > 0])}")
        print(f"Etiketli Piksel Sayısı: {np.sum(salinas_gt > 0)}")
        print(f"Etiketli Piksel Oranı: {np.sum(salinas_gt > 0) / (salinas_img.shape[0] * salinas_img.shape[1]) * 100:.2f}%")
    else:
        print("Salinas .mat dosyaları bulunamadı. Lütfen manuel olarak indirin.")
else:
    print("Salinas klasöründe .mat dosyası bulunamadı.")

## 10. Özet ve İstatistikler

In [ ]:
# Tüm veri setleri için özet tablo oluştur
summary_data = {
    'Dataset': ['Indian Pines', 'Pavia University', 'Salinas'],
    'Boyut (Piksel)': ['145×145', '610×340', '512×217'],
    'Bant Sayısı': [200, 103, 204],
    'Sınıf Sayısı': [16, 9, 16],
    'Toplam Piksel': [145*145, 610*340, 512*217]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("VERİ SETLERI ÖZET")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)

## 11. Spektral Bantların Görselleştirilmesi (İlk Yüklenen Veri Seti için)

In [ ]:
# Indian Pines'ın bazı spektral bantlarını görselleştir
if 'indian_img' in locals():
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Indian Pines - Seçilmiş Spektral Bantlar', fontsize=16, fontweight='bold')
    
    # Farklı bant indeksleri seç
    band_indices = [0, 50, 100, 150, 199, 99]
    
    for idx, ax in enumerate(axes.flat):
        band = band_indices[idx]
        im = ax.imshow(indian_img[:, :, band], cmap='viridis')
        ax.set_title(f'Bant {band}', fontweight='bold')
        ax.axis('off')
        plt.colorbar(im, ax=ax)
    
    plt.tight_layout()
    plt.show()
    print("Spektral bantlar başarıyla görselleştirildi.")

## 12. Sınıf Dağılımı (İlk Yüklenen Veri Seti için)

In [ ]:
# Indian Pines sınıf dağılımını görselleştir
if 'indian_gt' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Sınıf haritası
    im1 = axes[0].imshow(indian_gt, cmap='tab20')
    axes[0].set_title('Indian Pines - Sınıf Haritası', fontweight='bold', fontsize=12)
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0])
    
    # Sınıf sayısı
    unique_classes = np.unique(indian_gt)
    class_counts = [np.sum(indian_gt == c) for c in unique_classes[unique_classes > 0]]
    
    axes[1].bar(range(1, len(class_counts) + 1), class_counts, color='steelblue')
    axes[1].set_title('Indian Pines - Sınıf Dağılımı', fontweight='bold', fontsize=12)
    axes[1].set_xlabel('Sınıf No')
    axes[1].set_ylabel('Piksel Sayısı')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print("Sınıf dağılımı başarıyla görselleştirildi.")

## 13. Veri Kaydetme ve Export

In [ ]:
# Yüklü veri setlerini NumPy dosyaları olarak kaydet (hızlı erişim için)
processed_dir = os.path.join(BASE_DIR, 'processed')
os.makedirs(processed_dir, exist_ok=True)

datasets = {}

if 'indian_img' in locals():
    np.save(os.path.join(processed_dir, 'indian_pines_img.npy'), indian_img)
    np.save(os.path.join(processed_dir, 'indian_pines_gt.npy'), indian_gt)
    datasets['Indian Pines'] = {
        'shape': indian_img.shape,
        'saved': True
    }
    print("✓ Indian Pines verisi kaydedildi")

if 'pavia_img' in locals():
    np.save(os.path.join(processed_dir, 'pavia_university_img.npy'), pavia_img)
    np.save(os.path.join(processed_dir, 'pavia_university_gt.npy'), pavia_gt)
    datasets['Pavia University'] = {
        'shape': pavia_img.shape,
        'saved': True
    }
    print("✓ Pavia University verisi kaydedildi")

if 'salinas_img' in locals():
    np.save(os.path.join(processed_dir, 'salinas_img.npy'), salinas_img)
    np.save(os.path.join(processed_dir, 'salinas_gt.npy'), salinas_gt)
    datasets['Salinas'] = {
        'shape': salinas_img.shape,
        'saved': True
    }
    print("✓ Salinas verisi kaydedildi")

print(f"\n✓ Tüm veriler {processed_dir} klasöründe başarıyla kaydedildi.")